# ДЗ №1 — Анализ отзывов клиентов с LLM

## Цель проекта
Анализ отзывов: тональность, аспекты, резюме. Сравнение one-shot и workflow.

## 1. Настройка и подключение к OpenRouter

In [21]:
# 1. Установка библиотеки OpenAI (совместима с OpenRouter)
!pip install -q openai

# 2. Импорт необходимых модулей
import os
from openai import OpenAI

# 3. Установка API-ключа напрямую в переменную окружения (обходной путь)
#    ВАЖНО: перед сдачей работы эту строку нужно будет удалить!
os.environ["OPENROUTER_API_KEY"] = "# Ключ API вводится пользователем"

# 4. Подключение к OpenRouter
api_key = os.environ.get("OPENROUTER_API_KEY")
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

print("Ключ установлен:", api_key[:15] + "..." if api_key else "❌ НЕ НАЙДЕН")

Ключ установлен: sk-or-v1-bdcba6...


## 2. Модели Pydantic — структурированный вывод

In [23]:
# 1. Импорт библиотеки Pydantic для строгой валидации данных
from pydantic import BaseModel, Field, ValidationError
from typing import List, Literal
import json
import re

# 2. Модель для одного аспекта отзыва
class Aspect(BaseModel):
    """Один аспект, упомянутый в отзыве."""
    category: str = Field(description="Категория: доставка, качество, цена, сервис...")
    polarity: Literal["positive", "negative", "neutral"] = Field(description="Тональность аспекта")
    quote: str = Field(description="Цитата из отзыва")

# 3. Финальная модель анализа отзыва
class ReviewAnalysis(BaseModel):
    """Полный структурированный результат анализа отзыва."""
    sentiment: Literal["positive", "negative", "neutral", "mixed"]
    confidence: float = Field(ge=0.0, le=1.0)
    aspects: List[Aspect]
    summary: str
    action_recommendation: str

# 4. Проверка, что модели корректно определены
print(" Модели Pydantic загружены:")
print("   - Aspect:", list(Aspect.model_fields.keys()))
print("   - ReviewAnalysis:", list(ReviewAnalysis.model_fields.keys()))

 Модели Pydantic загружены:
   - Aspect: ['category', 'polarity', 'quote']
   - ReviewAnalysis: ['sentiment', 'confidence', 'aspects', 'summary', 'action_recommendation']


## 3. Baseline — один вызов LLM (one-shot)

In [22]:
# 1. Промпт для one-shot: просим LLM сделать всё за один вызов
ONE_SHOT_PROMPT = """Ты — аналитик отзывов клиентов. Проанализируй отзыв и верни ТОЛЬКО валидный JSON
без markdown-разметки и без пояснений, строго по схеме:

{{
  "sentiment": "positive" | "negative" | "neutral" | "mixed",
  "confidence": число от 0.0 до 1.0,
  "aspects": [
    {{"category": "категория", "polarity": "positive"|"negative"|"neutral", "quote": "цитата"}}
  ],
  "summary": "краткое резюме",
  "action_recommendation": "рекомендация"
}}

Отзыв:
{review}
"""

# 2. Функция безопасного парсинга JSON с повторными попытками
def safe_parse(raw: str, model_cls, max_retries: int = 0):
    """Пытается распарсить JSON. При ошибке — просит LLM исправить."""
    for attempt in range(max_retries + 1):
        try:
            cleaned = re.sub(r"^```json\s*|\s*```$", "", raw.strip(), flags=re.MULTILINE)
            data = json.loads(cleaned)
            return model_cls(**data)
        except (json.JSONDecodeError, ValidationError) as e:
            if attempt == max_retries:
                raise ValueError(f"Не удалось распарсить после {max_retries+1} попыток: {e}")
            # Просим LLM исправить ответ (с отключённым reasoning)
            fix_resp = client.chat.completions.create(
                model="nvidia/nemotron-3.5-super-vl:free",
                messages=[
                    {"role": "system", "content": "Верни ТОЛЬКО валидный JSON без markdown."},
                    {"role": "user", "content": f"Ошибка: {e}\nИсправь этот ответ:\n{raw}"},
                ],
                max_tokens=500,
                extra_body={"reasoning": {"enabled": False}},  # ← AJOUT
            )
            raw = fix_resp.choices[0].message.content

# 3. Реализация one-shot
def run_one_shot(review: str) -> dict:
    """Один вызов LLM — вся обработка за раз."""
    import time
    start = time.time()
    try:
        response = client.chat.completions.create(
            model="nvidia/nemotron-3.5-super-vl:free",
            messages=[{"role": "user", "content": ONE_SHOT_PROMPT.format(review=review)}],
            max_tokens=800,
            temperature=0,
            extra_body={"reasoning": {"enabled": False}},  # ← AJOUT
        )
        raw = response.choices[0].message.content
        if raw is None or raw.strip() == "":
            raise ValueError("LLM вернул пустой ответ")
        result = safe_parse(raw, ReviewAnalysis)
        elapsed = time.time() - start
        return {
            "success": True,
            "result": result.model_dump(),
            "tokens": response.usage.total_tokens,
            "time": elapsed,
            "calls": 1,
            "error": None,
        }
    except Exception as e:
        return {
            "success": False,
            "result": None,
            "tokens": 0,
            "time": time.time() - start,
            "calls": 1,
            "error": str(e),
        }

# 4. Быстрый тест на одном отзыве
test_review = "Доставка была очень быстрой, но товар пришёл поцарапанным. Поддержка ответила быстро."
res = run_one_shot(test_review)
print("✅ One-shot успешно" if res["success"] else f"❌ Ошибка: {res['error']}")
print(json.dumps(res["result"], ensure_ascii=False, indent=2) if res["success"] else "")
print(f"\n⏱ Время: {res['time']:.2f} с | Токены: {res['tokens']}")

✅ One-shot успешно
{
  "sentiment": "mixed",
  "confidence": 0.85,
  "aspects": [
    {
      "category": "доставка",
      "polarity": "positive",
      "quote": "Доставка была очень быстрой"
    },
    {
      "category": "товар",
      "polarity": "negative",
      "quote": "товар пришёл поцарапанным"
    },
    {
      "category": "поддержка",
      "polarity": "positive",
      "quote": "Поддержка ответила быстро"
    }
  ],
  "summary": "Быстрая доставка и поддержка, но товар повреждён.",
  "action_recommendation": "Оформить обмен или возврат поцарапанного товара и проверить упаковку на складе."
}

⏱ Время: 0.98 с | Токены: 372


## 4. Sequential workflow — 3 шага

In [24]:
# ============================================================
# WORKFLOW МНОГОШАГОВЫЙ (Sequential Pattern)
# ============================================================

import time

# ---------- Модели для промежуточных результатов ----------

class SentimentResult(BaseModel):
    sentiment: Literal["positive", "negative", "neutral", "mixed"]
    confidence: float = Field(ge=0.0, le=1.0)

class AspectsResult(BaseModel):
    aspects: List[Aspect]

class SummaryResult(BaseModel):
    summary: str
    action_recommendation: str


# ---------- Вспомогательная функция вызова LLM (с retry) ----------

MODEL_NAME = "nvidia/nemotron-3.5-super-vl:free"

def call_llm_json(prompt: str, max_tokens: int = 400) -> tuple:
    """Вызов LLM с retry и отключённым reasoning."""
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=0,
                extra_body={"reasoning": {"enabled": False}},  # отключаем reasoning
            )
            content = response.choices[0].message.content
            if content is None or content.strip() == "":
                print(f"⚠️ Пустой ответ (попытка {attempt+1}/3), повторяем...")
                time.sleep(1)
                continue
            return content, response.usage.total_tokens
        except Exception as e:
            print(f"⚠️ Ошибка (попытка {attempt+1}/3): {e}")
            if attempt == 2:
                raise
            time.sleep(1)
    raise ValueError("LLM вернул пустой ответ после 3 попыток")


# ---------- ШАГ 1 ----------
def step1_sentiment(review: str) -> tuple:
    prompt = f"""Проанализируй тональность отзыва. Верни ТОЛЬКО JSON:
{{"sentiment": "positive"|"negative"|"neutral"|"mixed", "confidence": 0.0-1.0}}

Отзыв: {review}
"""
    raw, tokens = call_llm_json(prompt, max_tokens=100)
    result = safe_parse(raw, SentimentResult)
    return result, tokens


# ---------- ШАГ 2 ----------
def step2_aspects(review: str, sentiment_ctx: str) -> tuple:
    prompt = f"""Извлеки аспекты из отзыва. Общая тональность: {sentiment_ctx}.
Верни ТОЛЬКО JSON: {{"aspects": [{{"category": "...", "polarity": "positive"|"negative"|"neutral", "quote": "..."}}]}}

Отзыв: {review}
"""
    raw, tokens = call_llm_json(prompt, max_tokens=400)
    result = safe_parse(raw, AspectsResult)
    return result, tokens


# ---------- ШАГ 3 ----------
def step3_summary(review: str, sentiment_ctx: str, aspects_ctx: list) -> tuple:
    aspects_str = json.dumps([a.model_dump() for a in aspects_ctx], ensure_ascii=False)
    prompt = f"""На основе анализа отзыва составь резюме и рекомендацию.
Тональность: {sentiment_ctx}
Аспекты: {aspects_str}
Отзыв: {review}

Верни ТОЛЬКО JSON: {{"summary": "...", "action_recommendation": "..."}}
"""
    raw, tokens = call_llm_json(prompt, max_tokens=300)
    result = safe_parse(raw, SummaryResult)
    return result, tokens


# ---------- ГЛАВНАЯ ФУНКЦИЯ ----------
def run_workflow(review: str) -> dict:
    start = time.time()
    total_tokens = 0
    calls = 0
    steps_log = []

    try:
        sentiment_res, t1 = step1_sentiment(review)
        total_tokens += t1; calls += 1
        steps_log.append({"step": 1, "ok": True, "tokens": t1})

        aspects_res, t2 = step2_aspects(review, sentiment_res.sentiment)
        total_tokens += t2; calls += 1
        steps_log.append({"step": 2, "ok": True, "tokens": t2})

        summary_res, t3 = step3_summary(review, sentiment_res.sentiment, aspects_res.aspects)
        total_tokens += t3; calls += 1
        steps_log.append({"step": 3, "ok": True, "tokens": t3})

        final = ReviewAnalysis(
            sentiment=sentiment_res.sentiment,
            confidence=sentiment_res.confidence,
            aspects=aspects_res.aspects,
            summary=summary_res.summary,
            action_recommendation=summary_res.action_recommendation,
        )

        return {
            "success": True, "result": final.model_dump(),
            "tokens": total_tokens, "time": time.time() - start,
            "calls": calls, "error": None, "steps": steps_log,
        }

    except Exception as e:
        return {
            "success": False, "result": None,
            "tokens": total_tokens, "time": time.time() - start,
            "calls": calls, "error": str(e), "steps": steps_log,
        }


# ---------- БЫСТРЫЙ ТЕСТ ----------
test_review = "Доставка была очень быстрой, но товар пришёл поцарапанным. Поддержка ответила быстро."
res = run_workflow(test_review)
print("✅ Workflow успешно" if res["success"] else f"❌ Ошибка: {res['error']}")
print(json.dumps(res["result"], ensure_ascii=False, indent=2) if res["success"] else "")
print(f"\n⏱ Время: {res['time']:.2f} с | Токены: {res['tokens']} | Вызовов: {res['calls']}")
print("Лог шагов:", res["steps"])

✅ Workflow успешно
{
  "sentiment": "mixed",
  "confidence": 0.85,
  "aspects": [
    {
      "category": "доставка",
      "polarity": "positive",
      "quote": "Доставка была очень быстрой"
    },
    {
      "category": "товар",
      "polarity": "negative",
      "quote": "товар пришёл поцарапанным"
    },
    {
      "category": "поддержка",
      "polarity": "positive",
      "quote": "Поддержка ответила быстро"
    }
  ],
  "summary": "Клиент высоко оценил скорость доставки и работу поддержки, но остался недоволен состоянием товара, который пришёл с царапинами.",
  "action_recommendation": "Оформить обмен или возврат поцарапанного товара, проверить контроль качества на складе и извиниться перед клиентом за дефект."
}

⏱ Время: 2.83 с | Токены: 596 | Вызовов: 3
Лог шагов: [{'step': 1, 'ok': True, 'tokens': 122}, {'step': 2, 'ok': True, 'tokens': 201}, {'step': 3, 'ok': True, 'tokens': 273}]


## 5. Тестовый набор — 10 примеров

In [25]:
# ============================================================
# ТЕСТОВЫЙ НАБОР: 10 отзывов (включая 2 нестандартных случая)
# ============================================================

TEST_REVIEWS = [
    # --- Стандартные случаи ---
    {"id": 1, "text": "Отличный товар! Всё пришло вовремя, упаковка целая. Очень доволен покупкой.",
     "type": "standard", "expected_sentiment": "positive"},
    {"id": 2, "text": "Ужасное качество. Куртка порвалась через неделю. Возврат не принимают, поддержка молчит.",
     "type": "standard", "expected_sentiment": "negative"},
    {"id": 3, "text": "Товар нормальный, ничего особенного. Цена соответствует качеству.",
     "type": "standard", "expected_sentiment": "neutral"},
    {"id": 4, "text": "Доставка быстрая, но товар пришёл поцарапанным. Поддержка ответила быстро и предложила замену.",
     "type": "standard", "expected_sentiment": "mixed"},
    {"id": 5, "text": "Очень удобное приложение, пользуюсь каждый день. Рекомендую всем друзьям!",
     "type": "standard", "expected_sentiment": "positive"},
    {"id": 6, "text": "Заказал 3 штуки, пришло только 2. Деньги за третью не вернули. Позор.",
     "type": "standard", "expected_sentiment": "negative"},
    {"id": 7, "text": "Цена хорошая, но качество среднее. В целом нормально, но не идеально.",
     "type": "standard", "expected_sentiment": "mixed"},
    {"id": 8, "text": "Быстрая доставка, вежливый курьер, товар соответствует описанию. Спасибо!",
     "type": "standard", "expected_sentiment": "positive"},
    # --- Нестандартные случаи ---
    {"id": 9, "text": "Ну спасибо, конечно. За такие деньги можно было бы и упаковать нормально. Но работает.",
     "type": "non_standard (сарказм)", "expected_sentiment": "mixed"},
    {"id": 10, "text": "👍",
     "type": "non_standard (очень короткий)", "expected_sentiment": "positive"},
]

print(f"✅ Загружено {len(TEST_REVIEWS)} тестовых отзывов")
print(f"   - Стандартных: {sum(1 for t in TEST_REVIEWS if t['type'] == 'standard')}")
print(f"   - Нестандартных: {sum(1 for t in TEST_REVIEWS if t['type'] != 'standard')}")

✅ Загружено 10 тестовых отзывов
   - Стандартных: 8
   - Нестандартных: 2


## 6. Результаты и сравнение

In [26]:
# ============================================================
# ЗАПУСК ОБОИХ РЕАЛИЗАЦИЙ (экономный режим)
# ============================================================
import pandas as pd
import time
import json

results_one_shot = []
results_workflow = []

for i, test in enumerate(TEST_REVIEWS):
    print(f"\n[{i+1}/10] Тест #{test['id']} ({test['type']})")
    print(f"Текст: {test['text'][:80]}...")

    # One-shot (1 запрос)
    r1 = run_one_shot(test["text"])
    results_one_shot.append({
        "id": test["id"], "type": test["type"], "expected": test["expected_sentiment"],
        "success": r1["success"],
        "sentiment": r1["result"]["sentiment"] if r1["success"] else None,
        "correct": r1["success"] and r1["result"]["sentiment"] == test["expected_sentiment"],
        "time": r1["time"], "tokens": r1["tokens"], "calls": r1["calls"], "error": r1["error"],
    })
    print(f"  One-shot: {'✅' if r1['success'] else '❌'} {r1['result']['sentiment'] if r1['success'] else r1['error']}")

    # Workflow (3 запроса)
    r2 = run_workflow(test["text"])
    results_workflow.append({
        "id": test["id"], "type": test["type"], "expected": test["expected_sentiment"],
        "success": r2["success"],
        "sentiment": r2["result"]["sentiment"] if r2["success"] else None,
        "correct": r2["success"] and r2["result"]["sentiment"] == test["expected_sentiment"],
        "time": r2["time"], "tokens": r2["tokens"], "calls": r2["calls"], "error": r2["error"],
    })
    print(f"  Workflow: {'✅' if r2['success'] else '❌'} {r2['result']['sentiment'] if r2['success'] else r2['error']}")

    # Сохраняем после каждого теста
    with open("results_backup.json", "w", encoding="utf-8") as f:
        json.dump({"one_shot": results_one_shot, "workflow": results_workflow}, f, ensure_ascii=False, indent=2)

    # Пауза 20 с между тестами
    if i < len(TEST_REVIEWS) - 1:
        print("⏳ Пауза 20 с...")
        time.sleep(20)

print("\n✅ Все тесты завершены !")


[1/10] Тест #1 (standard)
Текст: Отличный товар! Всё пришло вовремя, упаковка целая. Очень доволен покупкой....
  One-shot: ✅ positive
  Workflow: ✅ positive
⏳ Пауза 20 с...

[2/10] Тест #2 (standard)
Текст: Ужасное качество. Куртка порвалась через неделю. Возврат не принимают, поддержка...
  One-shot: ✅ negative
  Workflow: ✅ negative
⏳ Пауза 20 с...

[3/10] Тест #3 (standard)
Текст: Товар нормальный, ничего особенного. Цена соответствует качеству....
  One-shot: ✅ neutral
  Workflow: ✅ neutral
⏳ Пауза 20 с...

[4/10] Тест #4 (standard)
Текст: Доставка быстрая, но товар пришёл поцарапанным. Поддержка ответила быстро и пред...
  One-shot: ✅ mixed
  Workflow: ✅ mixed
⏳ Пауза 20 с...

[5/10] Тест #5 (standard)
Текст: Очень удобное приложение, пользуюсь каждый день. Рекомендую всем друзьям!...
  One-shot: ✅ positive
  Workflow: ✅ positive
⏳ Пауза 20 с...

[6/10] Тест #6 (standard)
Текст: Заказал 3 штуки, пришло только 2. Деньги за третью не вернули. Позор....
  One-shot: ✅ negative
  Wor

## 7. Сравнение one-shot и workflow


In [27]:
# ============================================================
# СРАВНИТЕЛЬНЫЙ АНАЛИЗ (с фильтрацией неудачных запусков)
# ============================================================
df1 = pd.DataFrame(results_one_shot)
df2 = pd.DataFrame(results_workflow)

ok1 = df1[df1["success"] == True]
ok2 = df2[df2["success"] == True]

comparison = pd.DataFrame({
    "Метрика": [
        "Доля успешных выполнений",
        "Accuracy (тональность, на успешных)",
        "Среднее время, с (на успешных)",
        "Среднее число LLM-вызовов (на успешных)",
        "Средний расход токенов (на успешных)",
    ],
    "One-shot": [
        f"{df1['success'].mean()*100:.0f}%",
        f"{ok1['correct'].mean()*100:.0f}%" if len(ok1) else "—",
        f"{ok1['time'].mean():.2f}" if len(ok1) else "—",
        f"{ok1['calls'].mean():.2f}" if len(ok1) else "—",
        f"{ok1['tokens'].mean():.0f}" if len(ok1) else "—",
    ],
    "Workflow": [
        f"{df2['success'].mean()*100:.0f}%",
        f"{ok2['correct'].mean()*100:.0f}%" if len(ok2) else "—",
        f"{ok2['time'].mean():.2f}" if len(ok2) else "—",
        f"{ok2['calls'].mean():.2f}" if len(ok2) else "—",
        f"{ok2['tokens'].mean():.0f}" if len(ok2) else "—",
    ],
})

print("СРАВНИТЕЛЬНАЯ ТАБЛИЦА\n")
print(comparison.to_string(index=False))

print("\nДЕТАЛИ ПО ТЕСТАМ\n")
details = pd.DataFrame({
    "ID": df1["id"], "Тип": df1["type"], "Ожидалось": df1["expected"],
    "One-shot": df1["sentiment"], "Workflow": df2["sentiment"],
    "OS OK": df1["correct"], "WF OK": df2["correct"],
})
print(details.to_string(index=False))

print("\nСЛУЧАИ РАСХОЖДЕНИЯ\n")
diff = details[details["One-shot"] != details["Workflow"]]
if len(diff) == 0:
    print("Расхождений нет.")
else:
    print(diff.to_string(index=False))

print("\nОШИБКИ (если есть)\n")
errs = pd.concat([
    df1[df1["success"] == False][["id", "error"]].assign(mode="One-shot"),
    df2[df2["success"] == False][["id", "error"]].assign(mode="Workflow"),
])
if len(errs) == 0:
    print("Ошибок нет.")
else:
    print(errs.to_string(index=False))

СРАВНИТЕЛЬНАЯ ТАБЛИЦА

                                Метрика One-shot Workflow
               Доля успешных выполнений     100%      90%
    Accuracy (тональность, на успешных)     100%     100%
         Среднее время, с (на успешных)     1.96     2.57
Среднее число LLM-вызовов (на успешных)     1.00     3.00
   Средний расход токенов (на успешных)      372      550

ДЕТАЛИ ПО ТЕСТАМ

 ID                           Тип Ожидалось One-shot Workflow  OS OK  WF OK
  1                      standard  positive positive positive   True   True
  2                      standard  negative negative negative   True   True
  3                      standard   neutral  neutral  neutral   True   True
  4                      standard     mixed    mixed    mixed   True   True
  5                      standard  positive positive positive   True   True
  6                      standard  negative negative negative   True   True
  7                      standard     mixed    mixed    mixed   True   True
  

## 8. Анализ и выводы

In [30]:

# АНАЛИЗ И ВЫВОДЫ


print("=" * 60)
print("АНАЛИЗ РЕЗУЛЬТАТОВ")
print("=" * 60)

succ_1 = df1['success'].mean() * 100
succ_2 = df2['success'].mean() * 100
print(f"\n1. Успешность выполнения:")
print(f"   One-shot: {succ_1:.0f}%  |  Workflow: {succ_2:.0f}%")

acc_1 = ok1['correct'].mean() * 100 if len(ok1) else 0
acc_2 = ok2['correct'].mean() * 100 if len(ok2) else 0
print(f"\n2. Точность (accuracy):")
print(f"   One-shot: {acc_1:.0f}%  |  Workflow: {acc_2:.0f}%")

time_1 = ok1['time'].mean() if len(ok1) else 0
time_2 = ok2['time'].mean() if len(ok2) else 0
print(f"\n3. Время выполнения:")
print(f"   One-shot: {time_1:.2f} с  |  Workflow: {time_2:.2f} с")

tok_1 = ok1['tokens'].mean() if len(ok1) else 0
tok_2 = ok2['tokens'].mean() if len(ok2) else 0
print(f"\n4. Расход токенов:")
print(f"   One-shot: {tok_1:.0f}  |  Workflow: {tok_2:.0f}")

err_1 = df1[df1['success'] == False]
err_2 = df2[df2['success'] == False]
print(f"\n5. Ошибки:")
print(f"   One-shot: {len(err_1)}  |  Workflow: {len(err_2)}")

print("\n" + "=" * 60)
print("ВЫВОД")
print("=" * 60)

if succ_2 > succ_1 and acc_2 >= acc_1:
    print("""
Workflow показал лучшие результаты:
- выше надёжность,
- не хуже точность,
- но больше времени и токенов.
""")
elif acc_2 > acc_1:
    print("""
Workflow точнее, но медленнее и дороже.
""")
else:
    print("""
Обе архитектуры показали сопоставимые результаты.
One-shot предпочтительнее из-за простоты и скорости.
""")

АНАЛИЗ РЕЗУЛЬТАТОВ

1. Успешность выполнения:
   One-shot: 100%  |  Workflow: 90%

2. Точность (accuracy):
   One-shot: 100%  |  Workflow: 100%

3. Время выполнения:
   One-shot: 1.96 с  |  Workflow: 2.57 с

4. Расход токенов:
   One-shot: 372  |  Workflow: 550

5. Ошибки:
   One-shot: 0  |  Workflow: 1

ВЫВОД

Обе архитектуры показали сопоставимые результаты.
One-shot предпочтительнее из-за простоты и скорости.

